In [1]:
import jax
from jax import lax
from jax import numpy as jnp
from jax import random as jrnd
import polars as pl

In [11]:
NGA_west2 = pl.scan_csv('NGA_West2/NGA_West2_Flatfile_RotD50_d005.csv', infer_schema_length = None)
print(NGA_west2.filter(pl.col('Record Sequence Number') == 2).collect())

shape: (1, 274)
┌──────────┬──────┬────────────┬──────┬───┬────────────────┬──────────────┬────────┬──────────┐
│ Record   ┆ EQID ┆ Earthquake ┆ YEAR ┆ … ┆ Late P-trigger ┆ Idirectivity ┆ Tp     ┆ Ry 2     │
│ Sequence ┆ ---  ┆ Name       ┆ ---  ┆   ┆ ---            ┆ ---          ┆ ---    ┆ ---      │
│ Number   ┆ i64  ┆ ---        ┆ i64  ┆   ┆ str            ┆ i64          ┆ f64    ┆ f64      │
│ ---      ┆      ┆ str        ┆      ┆   ┆                ┆              ┆        ┆          │
│ i64      ┆      ┆            ┆      ┆   ┆                ┆              ┆        ┆          │
╞══════════╪══════╪════════════╪══════╪═══╪════════════════╪══════════════╪════════╪══════════╡
│ 2        ┆ 2    ┆ Helena,    ┆ 1935 ┆ … ┆ -999           ┆ 0            ┆ -999.0 ┆ 0.155745 │
│          ┆      ┆ Montana-02 ┆      ┆   ┆                ┆              ┆        ┆          │
└──────────┴──────┴────────────┴──────┴───┴────────────────┴──────────────┴────────┴──────────┘


In [ ]:
NGA_west2 = pl.scan_csv('NGA_West2/NGA_West2_Flatfile_RotD50_d005.csv', infer_schema_length = None)
names = ['Mw', 'dip', 'rake', 'width', 'R_jb', 'R_rup', 'R_x', 'vs30', 'vs_flag', 'z1p0', 'z2p5', ' z_hyp', 'z_tor']

def pack_scenario(Mw:float, dip:float, rake:float, width:float,
                  R_jb:float, R_rup:float, R_x:float,
                  vs30:float, vs_flag:bool,
                  z1p0:float, z2p5:float, 
                  z_hyp:float, z_tor:float,
                  names:list = names):
        return dict(zip(names, locals()))

def NGA_scenario(RSN:int):
        assert RSN > 0 and RSN <= 21539, "Out of range."
        raw_scenario = NGA_west2.filter(pl.col('Record Sequence Number') == RSN).collect()
        EQID = raw_scenario.select('EQID').item()
        loc = raw_scenario.select(['Hypocenter Latitude (deg)', 'Hypocenter Longitude (deg)']).to_jax()

        Mw = 2 / 3 * jnp.log(raw_scenario.select('Mo (dyne.cm)').item()) / jnp.log(10) - 10.7
        
        dip, rake, width = 7, 8, 23
        R_jb, R_x = 39, 43
        # Top of rupture is z_tor, bottom is z_tor + (dip * (R_x - R_jb))
        z1, z2p5, z_hyp, z_tor = (88, 91), (90, 93), 16, 21
        for i in raw_scenario.columns:
                print(i)



NGA_scenario(5)

Record Sequence Number
EQID
Earthquake Name
YEAR
MODY
HRMN
Station Name
Station Sequence Number
Station ID  No.
Earthquake Magnitude
Magnitude Type
Magnitude Uncertainty: Kagan Model
Magnitude Uncertainty: Statistical
Magnitude Sample Size
Magnitude Uncertainty: Study Class
Mo (dyne.cm)
Strike (deg)
Dip (deg)
Rake Angle (deg)
Mechanism Based on Rake Angle
P-plunge (deg)
P-trend (deg)
T-plunge (deg)
T-trend (deg)
Hypocenter Latitude (deg)
Hypocenter Longitude (deg)
Hypocenter Depth (km)
Coseismic Surface Rupture: 1=Yes; 0=No;    -999=Unknown
Coseismic Surface Rupture (Including Inferred)
Basis for Inference of Surface Rupture
Finite Rupture Model: 1=Yes;  0=No
Depth to Top Of Fault Rupture Model
Fault Rupture Length for Calculation of Ry (km)
Fault Rupture Width (km)
Fault Rupture Area (km^2)
Avg Fault Disp (cm)
Rise Time (s)
Avg Slip Velocity (cm/s)
Static Stress Drop (bars)
Preferred Rupture Velocity (km/s)
Average Vr/Vs
Percent of Moment Release in the Top 5 Km of Crust
Existence of 